# 1. Importacion de librerias
Se importan las libreias necesarias

In [ ]:
import logging
logging.getLogger("torch.utils.flop_counter").setLevel(logging.ERROR)

In [ ]:
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from copy import deepcopy
from pathlib import Path

# ----------------------------

import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import torch

from keras import layers
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

print(f"GPU disponible: {'Si' if torch.cuda.is_available() else 'No'}")


# 2. Definicion de constantes necesarias

In [ ]:
# --------------------------------------------------------
# Carga de dataset
# --------------------------------------------------------
# Tope por clase 
N_PER_CLASS = None

# Tope por clase para entrenamiento
N_PER_CLASS_TRAIN = 100

# Direccion a buscar
DIR_DATASET = Path("data/PlantVillage")

CLASS_NAMES = sorted([dir.name for dir in DIR_DATASET.iterdir() if dir.is_dir()])
N_CLASS = len(CLASS_NAMES)

# Division
SPLIT = (0.70, 0.15, 0.15)

# Rescoluion imagen
IMG_SIZE = (224, 224, 3)

# --------------------------------------------------------
# Finetuning del backbone
# --------------------------------------------------------
FINETUNE = False

# --------------------------------------------------------
# Funcion objetivo
# --------------------------------------------------------
W_F1 = 0.7
W_ACC = 0.3
LAMBDA_GAP = 0.5
LAMBDA_LOSS = 0.1

FITNESS_FUNCTION = lambda f1, acc, gap, loss: W_F1 * f1 + W_ACC * acc - LAMBDA_GAP * gap - LAMBDA_LOSS * loss

# --------------------------------------------------------
# Algoritmo genetico
# --------------------------------------------------------

# Clase para los hiperparametros
class HyperParams:
    def __init__(
        self,
        alpha: float|None = None,
        batch: int|None = None,
        phi: str|None = None,
        rho: float|None = None
    ):
        self.alpha = alpha
        self.batch = batch
        self.phi = phi
        self.rho = rho

    def __getitem__(
        self, 
        key: str,
    ):
        return getattr(self, key)
    
    def __setitem__(
        self, 
        key: str, 
        value,
    ):
        setattr(self, key, value)

    def keys(
        self,
    ):
        return ["alpha", "batch", "phi", "rho"]

    def copy(
        self,
    ):
        return deepcopy(self)

# Rangos de los hiperparametros
ALPHA = (1e-4, 1e-2)
BATCH = (8, 16, 32, 64)
PHI = ("adam", "sgd", "rmsprop")
RHO = (0.0, 0.50)

# Tipo de cruzamiento por hiperparámetro:
UNIFORM_CROSS = ("batch", "phi")
BLEND_CROSSOVER = ("alpha", "rho")

# Poblacion
POP_SIZE = 12

# Probabilidad de mutacion:
MUTATION_RATE = 0.15

# Generaciones maximas
GEN_MAX = 20

# Paciencia en las generaciones
GEN_PAT = 5

# Individuos base en la seleccion
K_SELECT = 6

# Epocas para el AG
EPOCHS_FITNESS = 5

# Epocas para el testeo final
EPOCHS_TESTEO = 20

# Inicializacion con semilla
SEED = 83749215
random.seed(SEED)
np.random.seed(SEED)

# --------------------------------------------------------
# Modelos de CNN
# --------------------------------------------------------
N_CLASS = len(CLASS_NAMES)

OPTIMIZERS = {
    "adam": keras.optimizers.Adam,
    "sgd": keras.optimizers.SGD,
    "rmsprop": keras.optimizers.RMSprop,
}

MODELS = {
    "MobileNetV2": {
        "fn": keras.applications.MobileNetV2,
        "preprocess": keras.applications.mobilenet_v2.preprocess_input,
    },
    "EfficientNetB0": {
        "fn": keras.applications.EfficientNetB0,
        "preprocess": keras.applications.efficientnet.preprocess_input,
    },
}

# Auxiliares
_EVAL_COUNT, _EVAL_TIME = 0, 0
_FEAT_TRAIN, _FEAT_VAL, _FEAT_DIM = None, None, None

# 3. Carga de dataset y division
### Se define las constantes necesarias para este apartado


In [ ]:
print("="*60)
print("Clases presentes: ")
[print(f"   - {class_name}") for class_name in CLASS_NAMES]
print("="*60)

### Funcion para la carga del dataset

In [ ]:
def load_images(
    data_dir: Path,
):
    X, y = [], []

    for idx, cname in enumerate(CLASS_NAMES):

        files = sorted((data_dir / cname).glob("*.JPG"))

        if (N_PER_CLASS is not None and len(files) > N_PER_CLASS):
            files = random.sample(files, N_PER_CLASS)

        for p in files:
            img = Image.open(p).convert("RGB").resize(IMG_SIZE[:2])
            X.append(np.asarray(img, dtype=np.uint8))
            y.append(idx)
    
    return np.stack(X), np.array(y)


In [ ]:
# Visualizacion del total de imagenes
X_raw, y_all = load_images(
    DIR_DATASET,
)

print("="*60)
print(f"Total: {X_raw.shape}")
print("="*60)

plt.figure(figsize=(3 * N_CLASS, 3))
for idx, cname in enumerate(CLASS_NAMES):
    i = np.where(y_all == idx)[0][0]                 
    plt.subplot(1, N_CLASS, idx + 1)
    plt.imshow(X_raw[i])
    plt.title(f"{cname}\n(n={np.sum(y_all == idx)})")
    plt.axis("off")
plt.tight_layout()
plt.show()

### Funcion para la division del dataset

In [ ]:
train_frac, valid_frac, test_frac = SPLIT

X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, 
    y_all, 
    test_size=(test_frac + valid_frac), 
    stratify=y_all, 
    random_state=SEED,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, 
    y_temp, 
    test_size=test_frac/(test_frac + valid_frac), 
    stratify=y_temp, 
    random_state=SEED,
)
print("="*60)
print(f"Dataset completo: {X_raw.shape[0]}")
print(f"    Train: {X_train.shape[0]}")
print(f"    Valid: {X_val.shape[0]}")
print(f"    Test: {X_test.shape[0]}")
print("="*60)

### Limitacion del dataset para entrenamiento en AG y RS

In [ ]:
if N_PER_CLASS_TRAIN is not None:    
    keep_idx = []

    for c in range(N_CLASS):
        idx_c = [idx for idx, label in enumerate(y_train) if label == c]
        if len(idx_c) > N_PER_CLASS_TRAIN:
            idx_c = random.sample(list(idx_c), N_PER_CLASS_TRAIN)
        keep_idx.extend(idx_c)
    
    random.shuffle(keep_idx)


X_train_m, y_train_m = X_train[keep_idx], y_train[keep_idx]

print("="*60)
print(f"Dataset usado para el entrenamiento en AG y RS: {X_raw.shape[0]}")
print(f"    Train: {X_train_m.shape[0]:>4}  (tope {N_PER_CLASS_TRAIN}/clase)")
print(f"    Valid: {X_val.shape[0]:>4}  (completo)")
print(f"    Test : {X_test.shape[0]:>4}  (completo)")
print("-"*60)

### Hallar pesos de clase debido al desbalance

In [ ]:
cw = compute_class_weight("balanced", classes=np.arange(N_CLASS), y=y_train)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}

print("="*60)
[print(f"   - {class_name}: {CLASS_WEIGHT[idx]}") for idx, class_name in enumerate(CLASS_NAMES)]
print("="*60)

# 4. Funciones y clases necesarias para el Entrenamiento de CNN
### Funcion para generar un modelo

In [ ]:
def build_head(
    hiperparams: HyperParams,
):
    inputs = keras.Input(shape=(_FEAT_DIM,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dropout(hiperparams["rho"])(x)
    outputs = layers.Dense(N_CLASS, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=OPTIMIZERS[hiperparams["phi"]](learning_rate=hiperparams["alpha"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def build_model(
    backbone: str,
    hiperparams: HyperParams,
):
    base = MODELS[backbone]["fn"](
        input_shape=IMG_SIZE,
        include_top=False,
        weights="imagenet",
    )
    base.trainable = FINETUNE

    data_aug = keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.05),
        ],
        name="augment"
    )

    inputs = keras.Input(shape=IMG_SIZE)
    x = data_aug(inputs)
    x = layers.Lambda(MODELS[backbone]["preprocess"])(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(hiperparams["rho"])(x)
    outputs = layers.Dense(N_CLASS, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=OPTIMIZERS[hiperparams["phi"]](learning_rate=hiperparams["alpha"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model

# 5. Funciones y clases necesarias para el AG1

### Definicion de la estrutura del individuo
Se implementa una clase para cada invidiuo, la cual guarde su cromosoma, fitness y tenga metodos para el cruzamiento y mutaciones.

In [ ]:
class Individual:
    def __init__(
        self,
        chromosome: HyperParams|None = None,
    ):
        if chromosome is not None:
            self.chromosome = chromosome.copy()
        else:
            raise ValueError("Error: chromosome es nulo")
        
        self.fitness: float = -1.0

    def set_fitness(
        self,
        value,
    ):
        if isinstance(value, np.ndarray):
            value = value.item()
        self.fitness = float(value)

    def crossover(
        self,
        other,
    ):
        chromosome1 = HyperParams()
        chromosome2 = HyperParams()

        # Se hace un cruzamiento uniforme para los genes phi y batch:
        for param in UNIFORM_CROSS:
            if random.uniform(0, 1) <= 0.5:
                chromosome1[param] = self.chromosome[param]
                chromosome2[param] = other.chromosome[param]
            else:
                chromosome1[param] = other.chromosome[param]
                chromosome2[param] = self.chromosome[param]

        # Se hace un cruzamiento blend para los genes alpha y rho:
        for param in BLEND_CROSSOVER:
            alpha = random.uniform(0, 1)
            chromosome1[param] = alpha * self.chromosome[param] + (1 - alpha) * other.chromosome[param]
            chromosome2[param] = alpha * other.chromosome[param] + (1 - alpha) * self.chromosome[param]

        return Individual(chromosome1), Individual(chromosome2)
    
    def mutation(
        self,
    ):
        chromosome = self.chromosome.copy()

        for param in chromosome.keys():
            if random.uniform(0, 1) <= MUTATION_RATE:
                if param == "alpha":
                    chromosome[param] = random.uniform(*ALPHA)
                elif param == "batch":
                    chromosome[param] = random.choice(BATCH)
                elif param == "phi":
                    chromosome[param] = random.choice(PHI)
                elif param == "rho":
                    chromosome[param] = random.uniform(*RHO)

        return Individual(chromosome)

### Funcion para obtener el fitness de un cromosoma

In [ ]:
def compute_fitness(
    metrics: dict,
):
    f1_macro = metrics["f1_macro"]
    acc = metrics["val_acc"]
    gap = metrics["gap"]
    loss = metrics["val_loss"]

    fitness = FITNESS_FUNCTION(f1_macro, acc, gap, loss)

    return fitness

def get_fitness(
    backbone: str,
    chromosome: HyperParams,
):
    global _EVAL_COUNT, _EVAL_TIME
    keras.backend.clear_session()

    model = build_head(
        chromosome,
    )
    
    t0 = time.time()
    history = model.fit(
        _FEAT_TRAIN, y_train_m,
        validation_data=(_FEAT_VAL, y_val),
        epochs=EPOCHS_FITNESS,
        batch_size=chromosome["batch"],
        class_weight=CLASS_WEIGHT,
        verbose=0,
    )
    train_time = time.time() - t0

    y_prob = model.predict(
        _FEAT_VAL,
        verbose=0,
    )
    y_pred = np.argmax(y_prob, axis=1)
    
    f1_macro  = f1_score(y_val, y_pred, average="macro")
    val_acc   = float(history.history["val_accuracy"][-1])
    val_loss   = float(history.history["val_loss"][-1])
    train_acc = float(history.history["accuracy"][-1])
    gap       = gap = max(train_acc - val_acc, 0.0)

    metrics = {
        "f1_macro":   float(f1_macro),
        "val_acc":    val_acc,
        "val_loss":   val_loss,
        "train_acc":  train_acc,
        "gap":        float(gap),
        "train_time": float(train_time),
    }

    fitness = compute_fitness(metrics)

    _EVAL_COUNT += 1
    _EVAL_TIME  += train_time

    del model
    torch.cuda.empty_cache()

    print(f"alpha={chromosome['alpha']:.2e} batch={chromosome['batch']:>2} "
          f"phi={chromosome['phi']:>7} rho={chromosome['rho']:.2f} | "
          f"F1={f1_macro:.4f} acc={val_acc:.4f} loss={val_loss:.3f} gap={gap:+.3f} "
          f"t={train_time:5.1f}s  ->  fitness={fitness:.4f}")
    return fitness, metrics
    

### Funcion para evaluar una poblacion de individuos

In [ ]:
def eval_population(
    backbone: str,
    population: list[Individual],
):
    pop_size = len(population)

    for i in range(pop_size):
        if population[i].fitness == -1:
            fitness, metrics = get_fitness(
                backbone,
                population[i].chromosome,
            )
            population[i].set_fitness(fitness)
            population[i].metrics = metrics

### Funcion para inicializar una poblacion de individuos


In [ ]:
def init_population(
    size: int, 
):
    population = []
    
    for _ in range(size):
        alpha = random.uniform(*ALPHA)
        batch = random.choice(BATCH)
        phi = random.choice(PHI)
        rho = random.uniform(*RHO)

        chromosome = (alpha, batch, phi, rho)
        population.append(Individual(HyperParams(*chromosome)))
    
    return population


### Funcion para la seleccion de padres

In [ ]:
def select_parents(
    population: list[Individual],
    tournament_size: int,
):
    list_indiv = []

    x1 = np.random.permutation(len(population))
    y1 = x1[:tournament_size]

    for i in range(tournament_size):
        list_indiv.append(population[y1[i]].fitness)

    iParent1 = np.argmax(list_indiv)

    list_indiv = []
    x2 = np.delete(x1, iParent1)
    x2 = np.random.permutation(x2)
    y2 = x2[:tournament_size]

    for i in range(tournament_size):
        list_indiv.append(population[y2[i]].fitness)

    iParent2 = np.argmax(list_indiv)

    return population[y1[iParent1]], population[y2[iParent2]]

### Funcion para la seleccion de la poblacion

In [ ]:
def select_survivors(
    population: list[Individual],
    offspring: list[Individual],
    num_survivors: int,
):
    next_population = []
    population.extend(offspring)
    population.sort(key=lambda x: x.fitness, reverse=True)

    next_population = population[:num_survivors]
    
    return next_population

### Algoritmo genetico para encontrar soluciones

In [ ]:
def genetic_algo(
    backbone,
    population,
    n_gen = GEN_MAX,
    p_gen = GEN_PAT,
):
    
    pop_size = len(population)

    eval_population(
        backbone=backbone,
        population=population,
    )

    best = sorted(population, key=lambda x: x.fitness, reverse=True)[0]
    best_fitness = [best.fitness]

    ag_stats = [
        {
            "gen": 0, "best": float(best.fitness),
            "mean": float(np.mean([i.fitness for i in population])),
            "std":  float(np.std([i.fitness for i in population])),
        }
    ]

    print(f"Poblacion inicial, mejor fitness = {best_fitness[:1]}")

    no_improve = 0
    TOL = 1e-4

    for gen in range(n_gen):

        mating_pool = []

        for i in range(int(pop_size/2)):
            mating_pool.append(select_parents(
                population=population,
                tournament_size=K_SELECT,
            ))
        
        offspring = []
        
        for i in range(int(pop_size/2)):
            papa = mating_pool[i][0]
            mama = mating_pool[i][1]
            offspring.extend(papa.crossover(
                other=mama
            ))

        offspring = [child.mutation() for child in offspring]

        eval_population(
            backbone=backbone,
            population=offspring,
        )

        population = select_survivors(
            population=population,
            offspring=offspring,
            num_survivors=pop_size,
        )

        best = sorted(population, key=lambda x: x.fitness, reverse=True)[0]
        best_fitness.append(best.fitness)

        f = [i.fitness for i in population]
        ag_stats.append(
            {
                "gen": gen + 1, "best": float(best.fitness),
                "mean": float(np.mean(f)), "std": float(np.std(f))
            }
        )

        if best_fitness[-1] > best_fitness[-2] + TOL:
            no_improve = 0
            print(f"Generación {gen + 1}, mejor fitness = {best_fitness[-1]}")
        else:
            no_improve += 1
            print(f"Generación {gen + 1}, mejor fitness = {best_fitness[-1]:.4f}. (sin mejora {no_improve}/{p_gen})")
                  
            if no_improve >= p_gen:
                print(f"Early stopping: {p_gen} generaciones sin mejora.")
                break

    print(f"Mejor individuo en la ultima geneacion con fitness = {best_fitness[-1]}")
    return best, best_fitness, ag_stats

# 6. Funciones necesarias para RandomSearch
### Funcion RandomSearch

In [ ]:
def random_search(
    backbone: str,
    n_evals: int,
):
    population = init_population(n_evals)
    eval_population(
        backbone=backbone,
        population=population,
    )

    best = None
    best_so_far = []
    for indiv in population:
        if best is None or indiv.fitness > best.fitness:
            best = indiv
        best_so_far.append(best.fitness)


    print(f"\nRandom Search | mejor fitness = {best.fitness:.4f}")
    return best, best_so_far

# 7. Helpers para el testeo final
### Evaluar testeo

In [ ]:
def evaluate_test(
    backbone: str,
    chromosome: HyperParams,
    epochs: int = EPOCHS_TESTEO,
):
    keras.backend.clear_session()

    model = build_model(backbone, chromosome)
    model.fit(
        X_train, y_train, 
        epochs=epochs,
        batch_size=chromosome["batch"], 
        class_weight=CLASS_WEIGHT, 
        verbose=0
    )
    
    y_prob = model.predict(
        X_test,
        verbose=0,
    )
    y_pred = np.argmax(y_prob, axis=1)

    f1  = f1_score(y_test, y_pred, average="macro")
    acc = accuracy_score(y_test, y_pred)
    ll   = log_loss(y_test, y_prob, labels=list(range(N_CLASS)))
    conf = float(np.max(y_prob, axis=1).mean())

    del model
    torch.cuda.empty_cache()
    
    return f1, acc, ll, conf, y_pred

# Ex1 Funciones y clases necesarias para el Entrenamiento de CNN

In [ ]:
BACKBONE = "EfficientNetB0"

# Precomputacion
def build_feature_extractor(
    backbone: str,
):
    base = MODELS[backbone]["fn"](
        input_shape=IMG_SIZE,
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False                      # congelado -> features fijas

    inputs = keras.Input(shape=IMG_SIZE)
    x = layers.Lambda(MODELS[backbone]["preprocess"])(inputs)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, x)

# --- Precompute (una sola vez para la busqueda) ---
keras.backend.clear_session()
_feat_ext  = build_feature_extractor(BACKBONE)

_FEAT_TRAIN = _feat_ext.predict(X_train_m, batch_size=64, verbose=0)   # busqueda: reducido (300)
_FEAT_VAL   = _feat_ext.predict(X_val,     batch_size=64, verbose=0)   # validacion completa
_FEAT_DIM   = _FEAT_TRAIN.shape[1]

del _feat_ext
torch.cuda.empty_cache()
print(f"Features cacheadas -> train {_FEAT_TRAIN.shape} | val {_FEAT_VAL.shape} | dim {_FEAT_DIM}")

### Se realiza la prueba con la configuracion por defecto

In [ ]:
print("="*60)
print("ESTRATEGIA 3: Configuración por defecto")
print("="*60)
default_chr = HyperParams(0.001, 32, "adam", 0.2)
default_indiv = Individual(default_chr)

_def_fit, _def_met = get_fitness(BACKBONE, default_chr)
default_indiv.set_fitness(_def_fit)
default_indiv.metrics = _def_met

### Se realiza la prueba mediante el AG propuesto

In [ ]:
print("="*60)
print("Algoritmo Genético")
print("="*60)
np.random.seed(SEED); random.seed(SEED)

# Reinicio de contadores y medicion de eficiencia
_EVAL_COUNT = 0; _EVAL_TIME = 0.0
_t0 = time.time()

population = init_population(POP_SIZE)
ag_best, ag_curve, ag_stats = genetic_algo(BACKBONE, population)

ag_wall      = time.time() - _t0     
ag_evals     = _EVAL_COUNT           
ag_eval_time = _EVAL_TIME            
print(f"\nAG -> {ag_evals} evaluaciones | entren. {ag_eval_time:.1f}s | wall {ag_wall:.1f}s")

### Se realiza la prueba mediante RandomSearch

In [ ]:
n_evals = POP_SIZE * len(ag_curve)
print("="*60)
print(f"Random Search ({n_evals} evaluaciones)")
print("="*60)

# Reinicio de contadores y medicion de eficiencia
_EVAL_COUNT = 0; _EVAL_TIME = 0.0
_t0 = time.time()

rs_best, rs_curve = random_search(BACKBONE, n_evals)

rs_wall      = time.time() - _t0
rs_evals     = _EVAL_COUNT
rs_eval_time = _EVAL_TIME
print(f"\nRS -> {rs_evals} evaluaciones | entren. {rs_eval_time:.1f}s | wall {rs_wall:.1f}s")

### Resumen

In [ ]:
print(f"Resumen (fitness en VALIDACIÓN):")
print(f"    - AG     : {ag_best.fitness:.4f}")
print(f"    - Random : {rs_best.fitness:.4f}")
print(f"    - Default: {default_indiv.fitness:.4f}")

### Covergencia

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(np.arange(1, len(ag_curve)+1) * POP_SIZE, ag_curve, marker="o", label="Algoritmo Genético")

plt.plot(np.arange(1, len(rs_curve)+1), rs_curve, marker="s", alpha=0.7, label="Random Search")

plt.axhline(default_indiv.fitness, ls="--", color="gray", label="Default")

plt.xlabel("Número de evaluaciones (entrenamientos)")
plt.ylabel("Mejor fitness (F1-macro val)")
plt.title("Experimento 1: convergencia por estrategia (mismo presupuesto)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Comparativa de eficiencia en la busqueda (AG vs RS)

In [ ]:
def evals_to_best(curve, budget_per_point, tol=1e-9):
    """Presupuesto (num. de evaluaciones) hasta alcanzar por primera vez el mejor valor de la curva."""
    best = max(curve)
    for i, v in enumerate(curve):
        if v >= best - tol:
            return (i + 1) * budget_per_point
    return len(curve) * budget_per_point

def conv_auc(curve):
    """Area bajo la curva de convergencia con el presupuesto normalizado a [0, 1] (mayor = converge antes)."""
    c = np.asarray(curve, dtype=float)
    if len(c) == 1:
        return float(c[0])
    x = np.linspace(0.0, 1.0, len(c))
    return float(np.trapezoid(c, x))

# Presupuesto por punto: el AG evalua POP_SIZE por generacion; RS evalua 1 por punto
ag_budget_pt = POP_SIZE
rs_budget_pt = 1

eff_rows = [
    {
        "Estrategia": "Algoritmo Genético",
        "Evaluaciones": ag_evals,
        "Mejor_fitness": round(float(ag_best.fitness), 4),
        "Evals_hasta_mejor": evals_to_best(ag_curve, ag_budget_pt),
        "AUC_convergencia": round(conv_auc(ag_curve), 4),
        "Tiempo_entren_s": round(ag_eval_time, 1),
        "Wall_time_s": round(ag_wall, 1),
        "Fitness_por_eval": round(float(ag_best.fitness) / max(ag_evals, 1), 5),
    },
    {
        "Estrategia": "Random Search",
        "Evaluaciones": rs_evals,
        "Mejor_fitness": round(float(rs_best.fitness), 4),
        "Evals_hasta_mejor": evals_to_best(rs_curve, rs_budget_pt),
        "AUC_convergencia": round(conv_auc(rs_curve), 4),
        "Tiempo_entren_s": round(rs_eval_time, 1),
        "Wall_time_s": round(rs_wall, 1),
        "Fitness_por_eval": round(float(rs_best.fitness) / max(rs_evals, 1), 5),
    },
]
efficiency_df = pd.DataFrame(eff_rows)
print(efficiency_df.to_string(index=False))

# --- Grafico: convergencia con presupuesto NORMALIZADO (comparacion justa) ---
plt.figure(figsize=(8, 5))
ag_x = np.linspace(0, 1, len(ag_curve))
rs_x = np.linspace(0, 1, len(rs_curve))
plt.plot(ag_x, ag_curve, marker="o", label=f"AG (AUC={conv_auc(ag_curve):.3f})")
plt.plot(rs_x, rs_curve, marker="s", alpha=0.7, label=f"RS (AUC={conv_auc(rs_curve):.3f})")
plt.xlabel("Presupuesto normalizado")
plt.ylabel(f"Mejor fitness")
plt.title("Eficiencia de la búsqueda: convergencia a igual presupuesto")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Comparativa final de los mejores modelos en base a un entrenamiento completo y testeo

In [ ]:
strategies = {
    "Algoritmo Genético": ag_best.chromosome,
    "Random Search":      rs_best.chromosome,
    "Default":            default_chr,
}

rows, preds = [], {}
for name, chrom in strategies.items():
    f1, acc, ll, conf, y_pred = evaluate_test(BACKBONE, chrom)
    preds[name] = y_pred
    rows.append({
        "Estrategia": name,
        "alpha": chrom["alpha"], "batch": chrom["batch"],
        "phi": chrom["phi"], "rho": round(chrom["rho"], 3),
        "F1_macro_test": round(f1, 4), "Accuracy_test": round(acc, 4),
        "LogLoss_test": round(ll, 4), "Confianza_test": round(conf, 4),
    })

# Se ordena por F1 y, como desempate, por menor LogLoss (metrica continua)
results_df = pd.DataFrame(rows).sort_values(
    ["F1_macro_test", "LogLoss_test"], ascending=[False, True]
)
print(results_df.to_string(index=False))

### Matriz de confusion de la mejor estrategia

In [ ]:
best_name = results_df.iloc[0]["Estrategia"]
cm = confusion_matrix(y_test, preds[best_name])
ConfusionMatrixDisplay(cm, display_labels=[c.replace("Potato___", "") for c in CLASS_NAMES]).plot(cmap="Blues")
plt.title(f"Matriz de confusión en Test — {best_name}")
plt.tight_layout(); plt.show()

# End. Guardado final para siguientes experimentos

In [ ]:
import json, os

def _chr(c):
    return {"alpha": float(c["alpha"]), "batch": int(c["batch"]),
            "phi": str(c["phi"]), "rho": float(c["rho"])}

os.makedirs("results", exist_ok=True)

result = {
    "backbone": BACKBONE,
    "seed": SEED,
    "config": {
        "N_PER_CLASS": N_PER_CLASS, "N_PER_CLASS_TRAIN": N_PER_CLASS_TRAIN,
        "SPLIT": list(SPLIT), "FINETUNE": bool(FINETUNE),
        "W_F1": W_F1, "W_ACC": W_ACC, "LAMBDA_GAP": LAMBDA_GAP, "LAMBDA_LOSS": LAMBDA_LOSS,
        "POP_SIZE": POP_SIZE, "GEN_MAX": GEN_MAX, "GEN_PAT": GEN_PAT,
        "K_SELECT": K_SELECT, "MUTATION_RATE": MUTATION_RATE,
        "EPOCHS_FITNESS": EPOCHS_FITNESS, "EPOCHS_TESTEO": EPOCHS_TESTEO,
        "space": {"ALPHA": list(ALPHA), "BATCH": list(BATCH), "PHI": list(PHI), "RHO": list(RHO)},
    },

    "ag_curve": [float(x) for x in ag_curve],
    "ag_stats": globals().get("ag_stats"),
    "rs_curve": [float(x) for x in rs_curve],

    # Eficiencia de la busqueda (AG vs RS)
    "efficiency": efficiency_df.to_dict(orient="records"),

    "winners": {
        "AG":      {"chromosome": _chr(ag_best.chromosome),  "val_fitness": float(ag_best.fitness),
                    "val_metrics": getattr(ag_best, "metrics", None)},
        "RS":      {"chromosome": _chr(rs_best.chromosome),  "val_fitness": float(rs_best.fitness),
                    "val_metrics": getattr(rs_best, "metrics", None)},
        "Default": {"chromosome": _chr(default_chr),         "val_fitness": float(default_indiv.fitness),
                    "val_metrics": getattr(default_indiv, "metrics", None)},
    },

    "test": results_df.to_dict(orient="records"),

    "class_names": CLASS_NAMES,
    "y_test": [int(v) for v in y_test],
    "preds": {k: [int(v) for v in p] for k, p in preds.items()},
}

fname = f"results/{BACKBONE}_seed{SEED}.json"
with open(fname, "w") as f:
    json.dump(result, f, indent=2)
print(f"Guardado: {fname}")